# 🚢 Prétraitement et Transformation des Données — Dataset Titanic
**COT_GenAI 2026 — Semaine 2, Jour 2**

Séquence de prétraitement :
1. Gérer les doublons
2. Gérer les valeurs manquantes
3. Feature Engineering
4. Traiter les valeurs aberrantes
5. Standardisation / Normalisation
6. Encodage des variables catégorielles
7. Transformation de l'âge en groupes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Chargement du dataset
df = pd.read_csv('train.csv')

print(f'Dimensions initiales : {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

---
## 🌟 Exercice 1 : Détection et Suppression des Doublons
**Objectif** : Identifier et supprimer les lignes dupliquées dans le dataset.

In [ ]:
# Nombre de lignes avant suppression
rows_before = df.shape[0]
print(f'Nombre de lignes AVANT : {rows_before}')

# Identifier les doublons
duplicates = df.duplicated()
print(f'Nombre de doublons détectés : {duplicates.sum()}')

# Afficher les doublons s'il y en a
if duplicates.sum() > 0:
    print('Lignes dupliquées :')
    display(df[duplicates])

# Supprimer les doublons
df = df.drop_duplicates()

# Vérification
rows_after = df.shape[0]
print(f'Nombre de lignes APRÈS : {rows_after}')
print(f'Lignes supprimées : {rows_before - rows_after}')
# Le dataset Titanic est déjà propre → 0 doublon attendu

---
## 🌟 Exercice 2 : Gestion des Valeurs Manquantes
**Objectif** : Explorer et appliquer différentes stratégies selon la nature des colonnes.

In [ ]:
# Identifier les colonnes avec des valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Valeurs manquantes': missing,
    'Pourcentage (%)': missing_pct.round(2)
}).query('`Valeurs manquantes` > 0')

print('Colonnes avec des valeurs manquantes :')
display(missing_df)

In [ ]:
# Stratégie 1 : Imputation par la MÉDIANE pour 'Age'
# La médiane est robuste aux valeurs aberrantes, contrairement à la moyenne.
imputer_median = SimpleImputer(strategy='median')
df['Age'] = imputer_median.fit_transform(df[['Age']])

print(f"[Age] Valeurs manquantes après imputation par médiane : {df['Age'].isnull().sum()}")

In [ ]:
# Stratégie 2 : Remplissage par le MODE pour 'Embarked'
# Seulement 2 valeurs manquantes → on utilise la valeur la plus fréquente.
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(mode_embarked)

print(f"[Embarked] Mode utilisé : '{mode_embarked}'")
print(f"[Embarked] Valeurs manquantes restantes : {df['Embarked'].isnull().sum()}")

In [ ]:
# Stratégie 3 : SUPPRESSION de la colonne 'Cabin'
# Plus de 77% de valeurs manquantes → trop peu de données exploitables.
print(f"[Cabin] Pourcentage de valeurs manquantes : {df['Cabin'].isnull().mean()*100:.1f}%")
df = df.drop(columns=['Cabin'])
print("[Cabin] Colonne supprimée.")

# Vérification finale
total_missing = df.isnull().sum().sum()
print(f"\nValeurs manquantes totales restantes : {total_missing}")
print('✅ Aucune valeur manquante restante !' if total_missing == 0 else '⚠️ Des valeurs manquantes subsistent.')

---
## 🌟 Exercice 3 : Feature Engineering
**Objectif** : Créer de nouvelles features utiles à partir des colonnes existantes.

In [ ]:
# Nouvelle feature : Taille de la famille
# SibSp = frères/sœurs/conjoint à bord | Parch = parents/enfants à bord
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 pour le passager lui-même

# Nouvelle feature : Passager voyageant seul
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

print('FamilySize :')
print(df['FamilySize'].value_counts().sort_index())
print(f"\nIsAlone : {df['IsAlone'].value_counts().to_dict()}")

In [ ]:
# Extraction du titre depuis le nom
# Format du nom : "Nom, Titre. Prénom"
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
print('Titres extraits (bruts) :')
print(df['Title'].value_counts())

In [ ]:
# Regrouper les titres rares sous 'Rare' (moins de 10 occurrences)
rare_titles = df['Title'].value_counts()[df['Title'].value_counts() < 10].index
df['Title'] = df['Title'].replace(rare_titles, 'Rare')

# Harmoniser les équivalents
df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

print('Titres après regroupement :')
print(df['Title'].value_counts())

# Encodage avec LabelEncoder
le = LabelEncoder()
df['Title_encoded'] = le.fit_transform(df['Title'])
print(f"\nMapping LabelEncoder : {dict(zip(le.classes_, le.transform(le.classes_)))}")

---
## 🌟 Exercice 4 : Détection et Traitement des Valeurs Aberrantes
**Objectif** : Détecter et traiter les outliers dans `Age` et `Fare`.

In [ ]:
# Visualisation AVANT traitement
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Distributions AVANT traitement des outliers", fontsize=13)

df[['Age', 'Fare']].boxplot(ax=axes[0])
axes[0].set_title('Boxplot Age & Fare')

df['Fare'].hist(ax=axes[1], bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Distribution Fare')

plt.tight_layout()
plt.show()

In [ ]:
# Détection par méthode IQR
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

fare_lower, fare_upper = detect_outliers_iqr(df['Fare'])
age_lower, age_upper = detect_outliers_iqr(df['Age'])

print(f"[Fare] Limites IQR : [{fare_lower:.2f}, {fare_upper:.2f}]")
print(f"Outliers Fare : {((df['Fare'] < fare_lower) | (df['Fare'] > fare_upper)).sum()}")
print(f"\n[Age] Limites IQR : [{age_lower:.2f}, {age_upper:.2f}]")
print(f"Outliers Age : {((df['Age'] < age_lower) | (df['Age'] > age_upper)).sum()}")

In [ ]:
# Stratégie 1 : Capping au quantile 0.98 pour Fare
fare_cap = df['Fare'].quantile(0.98)
print(f"[Fare] Seuil de capping (quantile 0.98) : {fare_cap:.2f}")
df['Fare'] = df['Fare'].clip(upper=fare_cap)

# Stratégie 2 : Transformation logarithmique pour Fare (réduire l'asymétrie)
df['Fare_log'] = np.log1p(df['Fare'])  # log1p évite log(0)
print(f"[Fare] Skewness avant log : {df['Fare'].skew():.2f} | après log : {df['Fare_log'].skew():.2f}")

# Stratégie 3 : Capping pour Age
age_cap = df['Age'].quantile(0.98)
df['Age'] = df['Age'].clip(upper=age_cap)
print(f"\n[Age] Seuil de capping (quantile 0.98) : {age_cap:.2f}")

In [ ]:
# Visualisation APRÈS traitement
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Distributions APRÈS traitement des outliers", fontsize=13)

df[['Age', 'Fare']].boxplot(ax=axes[0])
axes[0].set_title('Boxplot Age & Fare')

df['Fare_log'].hist(ax=axes[1], bins=40, color='teal', edgecolor='white')
axes[1].set_title('Distribution Fare (log)')

plt.tight_layout()
plt.show()

---
## 🌟 Exercice 5 : Standardisation et Normalisation
**Objectif** : Mettre à l'échelle les features numériques après traitement des outliers.

- **StandardScaler** → mean=0, std=1 (pour distributions proches du normal)
- **MinMaxScaler** → plage [0,1] (pour distributions asymétriques ou bornées)

In [ ]:
# StandardScaler pour Age et FamilySize
standard_cols = ['Age', 'FamilySize']
scaler_std = StandardScaler()
df[standard_cols] = scaler_std.fit_transform(df[standard_cols])
print(f'StandardScaler appliqué sur : {standard_cols}')
display(df[standard_cols].describe().round(3))

# MinMaxScaler pour Fare et Fare_log
minmax_cols = ['Fare', 'Fare_log']
scaler_mm = MinMaxScaler()
df[minmax_cols] = scaler_mm.fit_transform(df[minmax_cols])
print(f'\nMinMaxScaler appliqué sur : {minmax_cols}')
display(df[minmax_cols].describe().round(3))

---
## 🌟 Exercice 6 : Encodage des Variables Catégorielles
**Objectif** : Finaliser l'encodage des variables catégorielles restantes.

In [ ]:
# Identifier les colonnes catégorielles restantes
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Colonnes catégorielles restantes : {cat_cols}')

for col in cat_cols:
    print(f"\n{col} : {df[col].unique()}")

In [ ]:
# One-Hot Encoding pour Sex, Embarked, Title
# drop_first=True pour éviter la multicolinéarité (dummy variable trap)
df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title'], drop_first=True)
print('One-Hot Encoding appliqué sur : Sex, Embarked, Title')

# Supprimer les colonnes textuelles non exploitables pour la modélisation
df = df.drop(columns=['Name', 'Ticket'], errors='ignore')
print("Colonnes 'Name' et 'Ticket' supprimées.")

# Vérification
remaining_cat = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nColonnes catégorielles restantes : {remaining_cat if remaining_cat else 'Aucune ✅'}")
print(f'\nDimensions : {df.shape}')
display(df.head())

---
## 🌟 Exercice 7 : Transformation de l'Âge en Groupes
**Objectif** : Créer des catégories d'âge avec `pd.cut()` et les encoder en one-hot.

In [ ]:
# On recharge l'âge original pour créer les groupes
# (Age a déjà été mis à l'échelle en Ex.5, on retravaille depuis les données brutes)
df_temp = pd.read_csv('train.csv')
df_temp['Age'] = SimpleImputer(strategy='median').fit_transform(df_temp[['Age']])
df_temp['Age'] = df_temp['Age'].clip(upper=df_temp['Age'].quantile(0.98))

# Créer des groupes d'âge avec pd.cut()
bins = [0, 12, 18, 60, 100]
labels = ['Enfant', 'Adolescent', 'Adulte', 'Senior']

df_temp['AgeGroup'] = pd.cut(df_temp['Age'], bins=bins, labels=labels, right=True)

print('Distribution des groupes d\'âge :')
print(df_temp['AgeGroup'].value_counts().sort_index())

In [ ]:
# Visualisation des groupes d'âge
plt.figure(figsize=(7, 4))
df_temp['AgeGroup'].value_counts().sort_index().plot(
    kind='bar', color=['#4e9af1', '#f18f4e', '#4ef169', '#f14e4e'], edgecolor='white'
)
plt.title("Répartition des passagers par groupe d'âge")
plt.xlabel("Groupe d'âge")
plt.ylabel("Nombre de passagers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# One-Hot Encoding des groupes d'âge avec pd.get_dummies()
age_dummies = pd.get_dummies(df_temp['AgeGroup'], prefix='Age')
print('Colonnes encodées :')
display(age_dummies.head())

# Fusionner avec le dataset principal
df = pd.concat([df.reset_index(drop=True), age_dummies.reset_index(drop=True)], axis=1)
print(f'\nDimensions finales après ajout des groupes d\'âge : {df.shape}')

---
## ✅ Résumé Final

In [ ]:
print(f'Dimensions finales : {df.shape}')
print(f'Valeurs manquantes : {df.isnull().sum().sum()}')
print(f'\nColonnes finales :')
for col in df.columns:
    print(f'  - {col} ({df[col].dtype})')

# Sauvegarder le dataset prétraité
df.to_csv('titanic_preprocessed.csv', index=False)
print('\n✅ Dataset prétraité sauvegardé : titanic_preprocessed.csv')